# 05 · Fine-tuning: full vs LoRA, Turkish and English

One task in two languages: rate a review from 1 to 5 stars, answering with a single digit in the chat format. Turkish reviews come from SentiTurca (e-commerce), English from Yelp; both sets have the same size. The models start from the SFT checkpoints (`sft-s50m-001`, `sft-m100m-001`).

Accuracy is measured **without generation**: one forward pass per review, and the prediction is the most likely of the tokens "1" … "5" (`glm_ladder/classify.py`).

| Step | Runs per model | What it decides |
|---|---:|---|
| Baselines | 0 | Majority class and zero-shot SFT model |
| LR sweep (Turkish) | 8 | Best LR for full fine-tuning and for LoRA rank 16, by Turkish validation macro-F1 |
| LoRA rank (Turkish) | 2 | Rank 4 and 64 at the LoRA learning rate found above |
| English | 2 | Full and LoRA rank 16 with the learning rates found in Turkish |
| Test | 0 | Selected runs, opened once: both languages' test sets, forgetting |

Every run is one epoch and a few minutes, so runs are not resumable: if the session dies, rerun the cells (finished runs are skipped, their receipts are in `/content/finetune` only while the runtime lives).

**Runtime:** *Select Kernel → Colab → New Colab Server → **A100***. About 2.5 hours for everything (measured: 31 runs, baselines and test).

## 1. GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Free the GPU

Must run before anything touches CUDA in this kernel.

In [ ]:
# In VS Code + Colab, "Restart Kernel" can leave the old kernel (and its compile workers)
# running and holding GPU memory, which later causes CUDA out-of-memory errors.
import os
import signal

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation

me = os.getpid()
ps = subprocess.run(["ps", "-eo", "pid,args"], capture_output=True, text=True).stdout.splitlines()[1:]
stale = []
for line in ps:
    pid_text, _, args = line.strip().partition(" ")
    pid = int(pid_text)
    if pid == me:
        continue
    if "colab_kernel_launcher" in args or ("compile_worker" in args and f"--parent={me}" not in args):
        stale.append((pid, args[:80]))
for pid, args in stale:
    os.kill(pid, signal.SIGKILL)
    print("killed stale process", pid, args)
print("stale processes:", len(stale))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 3. Get the code

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

## 4. Run all tests

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(result.stdout[-2500:], result.stderr[-1000:])
assert result.returncode == 0, "tests failed - fix before training"

## 5. Log in to Hugging Face

The token is typed into a hidden prompt and lives only in this session's memory.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

DATA_REPO = "Tbt-AI/glm-ladder-data"
RUNS_REPO = "Tbt-AI/glm-ladder-runs"

token = getpass("Hugging Face write token: ")
api = HfApi(token=token)
print("logged in as:", api.whoami()["name"])
for repo_id, repo_type in ((DATA_REPO, "dataset"), (RUNS_REPO, "model")):
    assert api.repo_info(repo_id, repo_type=repo_type).private, f"{repo_id} must be private"
    print("private:", repo_id)

## 6. Download and verify the data

The fine-tuning data (both languages) and, to measure forgetting, the SFT validation windows. Every file's SHA-256 is checked against its manifest.

In [ ]:
import json
import time

from huggingface_hub import snapshot_download

from glm_ladder.checkpoint import sha256_file

DATA_DIR = Path("/content/data")
started = time.perf_counter()
snapshot_download(repo_id=DATA_REPO, repo_type="dataset", local_dir=DATA_DIR, token=token,
                  allow_patterns=["finetune/*", "sft/manifest.json", "sft/*/val/*"])
print(f"downloaded in {time.perf_counter() - started:.0f} s")

tokenizer_receipt = json.loads((REPO_DIR / "artifacts" / "receipts" / "tokenizer-16k.json").read_text())
ft_manifest = json.loads((DATA_DIR / "finetune" / "manifest.json").read_text())
sft_manifest = json.loads((DATA_DIR / "sft" / "manifest.json").read_text())
for manifest in (ft_manifest, sft_manifest):
    assert manifest["tokenizer_sha256"] == tokenizer_receipt["sha256"], "tokenizer mismatch"
for language, splits in ft_manifest["languages"].items():
    for split, info in splits.items():
        for file, digest in info["sha256"].items():
            assert sha256_file(DATA_DIR / "finetune" / language / file) == digest, f"{language}/{file}"
        print(f"{language} {split:5s} {info['examples']:>7,} reviews  stars {info['stars']}")
for source, splits in sft_manifest["sources"].items():
    for file, digest in splits["val"]["sha256"].items():
        assert sha256_file(DATA_DIR / "sft" / source / "val" / file) == digest, f"sft {source}/{file}"
print("all files verified")

## 7. Sources, tasks and the two starting models

The starting weights are the final SFT checkpoints, checked against the SHA-256 in their `trainer.json`.

In [ ]:
import torch
from safetensors.torch import load_model

from glm_ladder import GLMLadderModel, ModelConfig
from glm_ladder.classify import LANGUAGES, evaluate_reviews, majority_baseline
from glm_ladder.code_eval import load_tasks
from glm_ladder.runs import RunStore
from glm_ladder.sft_data import SFTSource
from glm_ladder.tokenizer import load_tokenizer

SEQ_LEN = ft_manifest["seq_len"]
train = {lang: SFTSource(DATA_DIR / "finetune" / lang / "train") for lang in LANGUAGES}
val_windows = {lang: SFTSource(DATA_DIR / "finetune" / lang / "val") for lang in LANGUAGES}
sft_val = {f"sft_{name}": SFTSource(DATA_DIR / "sft" / name / "val") for name in sft_manifest["sources"]}


def read_jsonl(path):
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]
    return [row["text"] for row in rows], [row["stars"] for row in rows]


reviews = {(lang, split): read_jsonl(DATA_DIR / "finetune" / lang / f"{split}.jsonl")
           for lang in LANGUAGES for split in ("val", "test")}
train_stars = {lang: [int(star) for star, count in ft_manifest["languages"][lang]["train"]["stars"].items()
                      for _ in range(count)] for lang in LANGUAGES}
tokenizer = load_tokenizer(REPO_DIR / "tokenizer" / "glm-ladder-16k.json")
code_tasks = load_tasks(REPO_DIR / "evals" / "code_tasks.jsonl")
device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

BASES = {"s50m": "sft-s50m-001", "m100m": "sft-m100m-001"}
base_weights, base_states = {}, {}
for size, run in BASES.items():
    checkpoint = RunStore(api, RUNS_REPO, run).latest()
    def download(name):
        return Path(api.hf_hub_download(repo_id=RUNS_REPO, filename=f"runs/{run}/{checkpoint}/{name}", repo_type="model"))
    state = json.loads(download("trainer.json").read_text())
    weights = download("model.safetensors")
    assert sha256_file(weights) == state["model_sha256"], f"{run}: weights do not match their SHA-256"
    base_weights[size], base_states[size] = weights, {**state, "checkpoint": f"{RUNS_REPO}/runs/{run}/{checkpoint}"}
    print(f"{size}: {run} @ {checkpoint}")
for lang in LANGUAGES:
    print(f"{lang}: {train[lang].num_windows:,} train windows, {val_windows[lang].num_windows:,} val windows")

## 8. Baselines

- **Majority:** always the most frequent training label.
- **Zero-shot:** the SFT model asked the same question, before any fine-tuning. `label_mass` is the probability it puts on the five digits at all; if it is low, the model wanted to answer in words.

In [ ]:
def load_base(size):
    model = GLMLadderModel(ModelConfig.from_dict(base_states[size]["model_config"]))
    load_model(model, str(base_weights[size]), strict=True)
    return model.to(device).eval()


def review_metrics(model, split):
    return {lang: evaluate_reviews(model, tokenizer, *reviews[(lang, split)], lang, batch_size=64, device=device)
            for lang in LANGUAGES}


def show(name, result):
    print(f"{name:34s} " + "  ".join(f"{lang}: acc {m['accuracy']:.3f} F1 {m['macro_f1']:.3f} MAE {m['mae_stars']:.2f} "
                                   f"mass {m.get('label_mass', 1):.2f}" for lang, m in result.items()))


baselines = {"majority": {lang: majority_baseline(train_stars[lang], reviews[(lang, "val")][1]) for lang in LANGUAGES}}
show("majority (val)", baselines["majority"])
for size in BASES:
    model = load_base(size)
    baselines[f"zero-shot {size}"] = review_metrics(model, "val")
    show(f"zero-shot {size} (val)", baselines[f"zero-shot {size}"])
    del model
    torch.cuda.empty_cache()

## 9. One fine-tuning run

`finetune(size, lang, method, lr, rank)` trains one epoch on one language, then scores both languages' validation sets. The same function serves full fine-tuning (`rank=0`) and LoRA.

**Equal reviews per step.** A Turkish review is about 60 tokens, an English one about 190, so a 1024-token window holds ~16 Turkish but only ~5 English reviews. With the same number of windows per step, English would get 3.5× more (and smaller) steps from the same number of reviews. Instead the batch is 32 windows in Turkish and proportionally more in English (rounded to whole micro-batches), so both languages see about the same number of reviews per step and the same number of steps; English simply costs more tokens.

Fixed for every run: warmup 5% of the steps, cosine decay to 10%, no weight decay (decay pulls weights towards zero, which for LoRA means towards no update and for full fine-tuning is negligible in one short epoch), LoRA alpha 16 on attention and expert layers. `torch.compile` is off: compiling every new model would take longer than its training.

Validation during training: the label loss in the trained language (`val`), the other language's label loss and the SFT validation sets (`extra`, forgetting).

In [ ]:
import math
import shutil

from glm_ladder.lora import merge_lora, trainable_parameters
from glm_ladder.pretrain import PretrainConfig, run_pretraining
from glm_ladder.runs import MetricsLog
from dataclasses import asdict

GPU = torch.cuda.get_device_name()
WORKDIR = Path("/content/ft-runs")
OUT = Path("/content/finetune")
OUT.mkdir(exist_ok=True)
MICRO_BATCH = 16
reviews_per_window = {lang: ft_manifest["languages"][lang]["train"]["examples"] / train[lang].num_windows
                      for lang in LANGUAGES}
BATCH_ROWS = {lang: max(MICRO_BATCH, MICRO_BATCH * round(32 * reviews_per_window["tr"] / reviews_per_window[lang]
                                                         / MICRO_BATCH)) for lang in LANGUAGES}
for lang in LANGUAGES:
    print(f"{lang}: {reviews_per_window[lang]:.1f} reviews per window, batch {BATCH_ROWS[lang]} windows "
          f"(~{BATCH_ROWS[lang] * reviews_per_window[lang]:.0f} reviews), "
          f"{math.ceil(train[lang].num_windows / BATCH_ROWS[lang])} steps per epoch")
results = {}


def run_name(size, lang, method, lr, rank):
    tag = "full" if method == "full" else f"lora{rank}"
    return f"ft-{lang}-{size}-{tag}-lr{lr:.0e}"


def finetune(size, lang, method, lr, rank=0):
    name = run_name(size, lang, method, lr, rank)
    if (OUT / f"{name}.json").exists():                      # finished earlier in this session
        results[name] = json.loads((OUT / f"{name}.json").read_text())
        return results[name]
    steps = math.ceil(train[lang].num_windows / BATCH_ROWS[lang])
    cfg = PretrainConfig(run_name=name, total_tokens=train[lang].num_windows * SEQ_LEN, seq_len=SEQ_LEN,
                         batch_rows=BATCH_ROWS[lang], micro_batch=MICRO_BATCH, max_lr=lr,
                         warmup_steps=max(5, steps // 20), weight_decay=0.0, eval_every=max(1, steps // 4),
                         eval_windows=128, log_every=25, checkpoint_minutes=1e9, compile=False,
                         lora_rank=rank if method == "lora" else 0, lora_alpha=16.0, lora_targets="all")
    other = "en" if lang == "tr" else "tr"
    if (WORKDIR / name).exists():
        shutil.rmtree(WORKDIR / name)                        # a run interrupted earlier starts over
    result = run_pretraining(cfg, ModelConfig.from_dict(base_states[size]["model_config"]),
                             train_sources={lang: train[lang]}, val_sources={lang: val_windows[lang]},
                             weights={lang: 1.0}, store=None, workdir=WORKDIR, device=device,
                             init_from=base_weights[size], extra_val_sources={f"label_{other}": val_windows[other], **sft_val},
                             metadata={"commit": commit, "stage": "finetune", "init_from": base_states[size]["checkpoint"]},
                             log=print)
    model = result["model"]
    trained = trainable_parameters(model)
    merge_lora(model)
    rows = MetricsLog(WORKDIR / name / "metrics.jsonl").rows()
    final = [row for row in rows if "eval" in row][-1]
    receipt = {
        "schema_version": "1.0", "kind": "finetune_run", "run_name": name, "commit": commit, "gpu": GPU,
        "size": size, "language": lang, "method": method, "lora_rank": cfg.lora_rank, "max_lr": lr,
        "init_from": base_states[size]["checkpoint"], "trainable_parameters": trained,
        "total_parameters": model.parameter_counts()["total"], "steps": result["step"],
        "tokens_seen": result["tokens_seen"],
        "train_seconds": round(sum(row["seconds"] for row in rows if "seconds" in row), 1),
        "final_eval": final["eval"], "final_eval_extra": final["eval_extra"],
        "val": review_metrics(model, "val"), "config": asdict(cfg),
    }
    (OUT / f"{name}.json").write_text(json.dumps(receipt, indent=2) + "\n")
    results[name] = receipt
    show(name, receipt["val"])
    del model, result
    torch.cuda.empty_cache()
    return receipt

## 10. Smoke test

The smallest run, to check the whole path before spending time: `s50m`, Turkish, LoRA rank 16. Watch the first loss (about ln 16384 ≈ 9.7 would mean the template is wrong; a well-formatted SFT model starts far lower) and the validation scores after one epoch.

In [ ]:
smoke = finetune("s50m", "tr", "lora", 3e-4, rank=16)
print(json.dumps({k: smoke[k] for k in ("trainable_parameters", "steps", "train_seconds", "final_eval")}, indent=2))

## 11. Learning-rate sweep (Turkish)

Full fine-tuning changes every weight, so it needs a small learning rate (SFT used 1e-4); LoRA's update starts at zero and is scaled by alpha / rank, so it usually needs a larger one. Four values each, chosen by **Turkish validation macro-F1** (fixed before seeing any result). The test sets stay closed. (The first run of this study used three values per grid; all four winners were at the top edge, so the grids were extended by one value before the test sets were opened. See PLAN.md.)

In [ ]:
FULL_LRS = (1e-5, 3e-5, 1e-4, 3e-4)
LORA_LRS = (1e-4, 3e-4, 1e-3, 3e-3)
best_lr = {}
for size in BASES:
    for method, lrs in (("full", FULL_LRS), ("lora", LORA_LRS)):
        scores = {lr: finetune(size, "tr", method, lr, rank=16 if method == "lora" else 0)["val"]["tr"]["macro_f1"]
                  for lr in lrs}
        best_lr[(size, method)] = max(scores, key=scores.get)
        print(f"==> {size} {method}: macro-F1 by LR {scores} -> best {best_lr[(size, method)]:.0e}")
        if best_lr[(size, method)] in (lrs[0], lrs[-1]):
            print("    note: the best LR is at the edge of the grid; a wider grid might do better")

## 12. LoRA rank (Turkish)

Rank 4 and 64 at each model's best LoRA learning rate (rank 16 is already trained). Because the update is scaled by alpha / rank, a larger rank does not by itself make the update larger.

In [ ]:
for size in BASES:
    for rank in (4, 64):
        finetune(size, "tr", "lora", best_lr[(size, "lora")], rank=rank)

## 13. English

Full fine-tuning and LoRA rank 16 with the learning rates found in Turkish, so the only change is the language.

In [ ]:
for size in BASES:
    finetune(size, "en", "full", best_lr[(size, "full")])
    finetune(size, "en", "lora", best_lr[(size, "lora")], rank=16)

## 14. Validation table

In [ ]:
keys = ["trainable_parameters", "train_seconds"]
print("| run | trainable | seconds | tr acc | tr F1 | tr MAE | en acc | en F1 | en MAE |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for name, r in sorted(results.items()):
    cells = [f"{r['trainable_parameters']:,}", f"{r['train_seconds']:.0f}"]
    for lang in LANGUAGES:
        m = r["val"][lang]
        cells += [f"{m['accuracy']:.3f}", f"{m['macro_f1']:.3f}", f"{m['mae_stars']:.2f}"]
    print(f"| {name} | " + " | ".join(cells) + " |")

## 15. Test, opened once

For each model: the zero-shot SFT model and the four selected runs (Turkish and English, full and LoRA rank 16 at the selected LR). Each is scored on **both** languages' test sets (cross-lingual transfer), plus forgetting: the reference-solution loss on the 40 coding tasks in chat mode (`scripts/run_code_eval.py --loss-only`; lower is better), next to the SFT validation losses from training.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "scripts"))
from run_code_eval import evaluate as code_evaluate   # scripts/run_code_eval.py
from glm_ladder.lora import apply_lora


def code_reference_loss(model):
    return code_evaluate(model, tokenizer, code_tasks, mode="chat", loss_only=True, log=lambda _: None)[1]["reference_loss"]


def load_finetuned(name):
    r = results[name]
    directory = sorted((WORKDIR / name).glob("step-*"))[-1]
    model = GLMLadderModel(ModelConfig.from_dict(base_states[r["size"]]["model_config"]))
    if r["lora_rank"]:
        apply_lora(model, rank=r["lora_rank"], alpha=r["config"]["lora_alpha"], targets=r["config"]["lora_targets"])
    load_model(model, str(directory / "model.safetensors"), strict=True)
    merge_lora(model)
    return model.to(device).eval()


test = {"majority": {lang: majority_baseline(train_stars[lang], reviews[(lang, "test")][1]) for lang in LANGUAGES}}
selected = {}
for size in BASES:
    model = load_base(size)
    test[f"zero-shot {size}"] = {**review_metrics(model, "test"), "code_reference_loss": code_reference_loss(model)}
    del model
    for lang in LANGUAGES:
        for method in ("full", "lora"):
            name = run_name(size, lang, method, best_lr[(size, method)], 16 if method == "lora" else 0)
            selected[name] = results[name]
            model = load_finetuned(name)
            test[name] = {**review_metrics(model, "test"), "code_reference_loss": code_reference_loss(model)}
            del model
            torch.cuda.empty_cache()
for name, result in test.items():
    show(name + " (test)", {lang: result[lang] for lang in LANGUAGES})
    if "code_reference_loss" in result:
        print(f"{'':34s} code reference loss {result['code_reference_loss']:.3f}")

## 16. Upload

Receipts of every run, the baselines and the test results go to the runs repo under `finetune/`. The weights of the selected runs go to `runs/<run_name>/` like any other run (without the optimizer state, which is only needed to resume), so `scripts/sample.py`, `talk.py` and `run_code_eval.py` can load them; LoRA runs are merged on load.

In [ ]:
summary = {"schema_version": "1.0", "kind": "finetune_summary", "commit": commit, "gpu": GPU,
           "data": {lang: {split: {k: v for k, v in info.items() if k != "sha256"} for split, info in splits.items()}
                    for lang, splits in ft_manifest["languages"].items()},
           "best_lr": {f"{size}-{method}": lr for (size, method), lr in best_lr.items()},
           "baselines_val": baselines, "test": test, "selected": sorted(selected)}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
api.upload_folder(folder_path=str(OUT), path_in_repo="finetune", repo_id=RUNS_REPO, repo_type="model",
                  commit_message=f"fine-tuning receipts, commit {commit}")
for name in selected:
    directory = sorted((WORKDIR / name).glob("step-*"))[-1]
    (directory / "optimizer.pt").unlink(missing_ok=True)
    RunStore(api, RUNS_REPO, name).push(directory, metrics=WORKDIR / name / "metrics.jsonl", milestone=True)
    print("uploaded", name)

## 17. Done

Close the runtime (the A100 keeps using units while idle). Locally:

```bash
python scripts/fetch_evals.py --pattern "finetune/*.json" --to artifacts/finetune
```